# 🎬 Video Deepfake Detection Model Training

This notebook trains a **video-specific deepfake detection model** using temporal analysis.

## Architecture
- **Backbone**: EfficientNet-B0 (pretrained) for frame features
- **Temporal Module**: LSTM to capture temporal inconsistencies
- **Output**: Binary classification (real/fake)

## Datasets Supported
- FaceForensics++ (FF++)
- Deepfake Detection Challenge (DFDC)
- Celeb-DF

---

## 1. Setup & Dependencies

In [1]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python-headless facenet-pytorch av pillow tqdm matplotlib scikit-learn seaborn

Looking in indexes: https://download.pytorch.org/whl/cu118


In [2]:
# Configure Kaggle credentials
import os
os.environ['KAGGLE_USERNAME'] = 'prathamcu823'
os.environ['KAGGLE_KEY'] = '576fb3d57e703f4f744ba3985d6e6b44'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"prathamcu823","key":"576fb3d57e703f4f744ba3985d6e6b44"}')
!chmod 600 /root/.kaggle/kaggle.json
print(" Kaggle configured!")

 Kaggle configured!


In [3]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from facenet_pytorch import MTCNN
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: Tesla T4
Memory: 15.83 GB


## 2. Download Dataset

Choose one of the following datasets:

In [4]:
# Option A: FaceForensics++ (requires academic access)
# Visit: https://github.com/ondyari/FaceForensics

# Option B: DFDC subset from Kaggle
!kaggle competitions download -c deepfake-detection-challenge
!unzip -q deepfake-detection-challenge.zip -d ./data/dfdc

deepfake-detection-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)
replace ./data/dfdc/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [10]:
!mkdir -p ./data/dfdc && unzip -q deepfake-detection-challenge.zip -d ./data/dfdc && ls -la ./data/dfdc/

replace ./data/dfdc/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [5]:
# Option C: Celeb-DF v2 (smaller, good for testing)
# Download from: https://github.com/yuezunli/celeb-deepfakeforensics

# Set your data path here
DATA_ROOT = "./data/dfdc"  # Change to your dataset path

## 3. Configuration

In [6]:
# Training configuration
class Config:
    # Data
    data_root = DATA_ROOT
    frames_per_video = 16          # Number of frames to sample per video
    frame_size = 224               # Input image size
    
    # Model
    lstm_hidden_size = 512         # LSTM hidden dimension
    lstm_num_layers = 2            # Number of LSTM layers
    dropout = 0.3                  # Dropout rate
    
    # Training
    batch_size = 8                 # Reduce if OOM
    num_epochs = 20
    learning_rate = 1e-4
    weight_decay = 1e-5
    patience = 5                   # Early stopping patience
    
    # Paths
    checkpoint_dir = "./checkpoints"
    model_name = "video_deepfake_detector"

config = Config()
os.makedirs(config.checkpoint_dir, exist_ok=True)

## 4. Video Dataset Class

In [7]:
class VideoDeepfakeDataset(Dataset):
    """
    Dataset for loading video frames with face detection.
    Samples N frames uniformly from each video.
    """
    
    def __init__(self, video_paths, labels, config, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.config = config
        self.transform = transform
        
        # Face detector
        self.mtcnn = MTCNN(
            image_size=config.frame_size,
            margin=40,
            keep_all=False,
            device=device
        )
    
    def __len__(self):
        return len(self.video_paths)
    
    def extract_frames(self, video_path):
        """Extract N frames uniformly from video with face detection."""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames == 0:
            cap.release()
            return None
        
        # Sample frame indices uniformly
        indices = np.linspace(0, total_frames - 1, self.config.frames_per_video, dtype=int)
        
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            
            if not ret:
                continue
            
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            # Detect and crop face
            try:
                face = self.mtcnn(pil_image)
                if face is not None:
                    frames.append(face)
                else:
                    # Fallback: resize whole frame
                    resized = pil_image.resize((self.config.frame_size, self.config.frame_size))
                    tensor = transforms.ToTensor()(resized)
                    frames.append(tensor)
            except Exception:
                continue
        
        cap.release()
        return frames if len(frames) > 0 else None
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        
        frames = self.extract_frames(video_path)
        
        if frames is None or len(frames) < self.config.frames_per_video // 2:
            # Return zeros if extraction failed
            frames_tensor = torch.zeros(self.config.frames_per_video, 3, 
                                        self.config.frame_size, self.config.frame_size)
        else:
            # Pad or truncate to exact frame count
            while len(frames) < self.config.frames_per_video:
                frames.append(frames[-1])  # Repeat last frame
            frames = frames[:self.config.frames_per_video]
            frames_tensor = torch.stack(frames)
        
        # Normalize
        normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        frames_tensor = torch.stack([normalize(f) for f in frames_tensor])
        
        return frames_tensor, torch.tensor(label, dtype=torch.long)

## 5. Video Deepfake Detection Model

In [8]:
class VideoDeepfakeDetector(nn.Module):
    """
    Video deepfake detector with EfficientNet backbone + LSTM temporal modeling.
    
    Architecture:
    1. EfficientNet extracts features from each frame
    2. LSTM captures temporal inconsistencies across frames
    3. FC layer outputs binary classification
    """
    
    def __init__(self, config):
        super(VideoDeepfakeDetector, self).__init__()
        
        # Frame-level feature extractor (EfficientNet-B0)
        efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(efficientnet.children())[:-1])
        self.backbone_dim = 1280  # EfficientNet-B0 output dimension
        
        # Freeze early layers for transfer learning
        for param in list(self.backbone.parameters())[:-20]:
            param.requires_grad = False
        
        # Temporal modeling (LSTM)
        self.lstm = nn.LSTM(
            input_size=self.backbone_dim,
            hidden_size=config.lstm_hidden_size,
            num_layers=config.lstm_num_layers,
            batch_first=True,
            dropout=config.dropout if config.lstm_num_layers > 1 else 0,
            bidirectional=True
        )
        
        # Classification head
        lstm_output_size = config.lstm_hidden_size * 2  # Bidirectional
        self.classifier = nn.Sequential(
            nn.Dropout(config.dropout),
            nn.Linear(lstm_output_size, 256),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(256, 2)  # 2 classes: real, fake
        )
    
    def forward(self, x):
        """
        x: (batch_size, num_frames, 3, H, W)
        """
        batch_size, num_frames, C, H, W = x.shape
        
        # Reshape for backbone: (batch * frames, C, H, W)
        x = x.view(batch_size * num_frames, C, H, W)
        
        # Extract frame features
        features = self.backbone(x)  # (batch * frames, 1280, 1, 1)
        features = features.squeeze(-1).squeeze(-1)  # (batch * frames, 1280)
        
        # Reshape back: (batch, frames, 1280)
        features = features.view(batch_size, num_frames, -1)
        
        # Temporal modeling
        lstm_out, _ = self.lstm(features)  # (batch, frames, hidden*2)
        
        # Use last hidden state
        final_hidden = lstm_out[:, -1, :]  # (batch, hidden*2)
        
        # Classification
        output = self.classifier(final_hidden)
        
        return output

# Initialize model
model = VideoDeepfakeDetector(config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model parameters: 15,262,898


## 6. Load and Prepare Data

In [11]:
def load_dfdc_data(data_root, max_videos=1000):
    """
    Load DFDC dataset metadata.
    Adjust this function based on your dataset structure.
    """
    import json
    
    video_paths = []
    labels = []
    
    # DFDC structure: train_sample_videos/metadata.json
    metadata_path = os.path.join(data_root, "train_sample_videos", "metadata.json")
    
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        for video_name, info in list(metadata.items())[:max_videos]:
            video_path = os.path.join(data_root, "train_sample_videos", video_name)
            if os.path.exists(video_path):
                video_paths.append(video_path)
                labels.append(1 if info['label'] == 'FAKE' else 0)
    else:
        # Fallback: scan directory
        print("Metadata not found. Scanning directory...")
        for subdir in ['real', 'fake', 'REAL', 'FAKE', 'Real', 'Fake']:
            dir_path = os.path.join(data_root, subdir)
            if os.path.exists(dir_path):
                label = 1 if 'fake' in subdir.lower() else 0
                for video_file in os.listdir(dir_path)[:max_videos // 2]:
                    if video_file.endswith(('.mp4', '.avi', '.mov')):
                        video_paths.append(os.path.join(dir_path, video_file))
                        labels.append(label)
    
    print(f"Loaded {len(video_paths)} videos")
    print(f"  Real: {labels.count(0)}, Fake: {labels.count(1)}")
    
    return video_paths, labels

# Load data
video_paths, labels = load_dfdc_data(config.data_root)

Loaded 400 videos
  Real: 77, Fake: 323


In [12]:
from sklearn.model_selection import train_test_split

# Split into train/val/test
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    video_paths, labels, test_size=0.3, stratify=labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# Create datasets
train_dataset = VideoDeepfakeDataset(train_paths, train_labels, config)
val_dataset = VideoDeepfakeDataset(val_paths, val_labels, config)
test_dataset = VideoDeepfakeDataset(test_paths, test_labels, config)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

Train: 280, Val: 60, Test: 60


## 7. Training Loop

In [13]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for frames, labels in pbar:
        frames, labels = frames.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for frames, labels in tqdm(loader, desc="Validating"):
            frames, labels = frames.to(device), labels.to(device)
            
            outputs = model(frames)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    accuracy = 100. * correct / total
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    
    return total_loss / len(loader), accuracy, auc, all_preds, all_labels

In [ ]:
# Training loop with early stopping
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

print("Starting training...")
print("=" * 60)

for epoch in range(config.num_epochs):
    print(f"\nEpoch {epoch+1}/{config.num_epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    
    # Validate
    val_loss, val_acc, val_auc, _, _ = validate(model, val_loader, criterion)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Log history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
            'val_auc': val_auc,
        }, os.path.join(config.checkpoint_dir, f'{config.model_name}_best.pth'))
        print("  ✓ Saved best model")
    else:
        patience_counter += 1
        if patience_counter >= config.patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("\nTraining complete!")

Starting training...

Epoch 1/20


Training:  74%|███████▍  | 26/35 [43:34<11:49, 78.84s/it, loss=0.2337, acc=79.81%]  

## 8. Plot Training Progress

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()

# AUC
axes[2].plot(history['val_auc'], label='Validation AUC', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC-ROC')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(config.checkpoint_dir, 'training_curves.png'), dpi=150)
plt.show()

## 9. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(config.checkpoint_dir, f'{config.model_name}_best.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Test evaluation
test_loss, test_acc, test_auc, test_preds, test_labels = validate(model, test_loader, criterion)

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test AUC-ROC: {test_auc:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Real', 'Fake']))

In [ ]:
# Confusion Matrix
import seaborn as sns

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Video Deepfake Detection')
plt.savefig(os.path.join(config.checkpoint_dir, 'confusion_matrix.png'), dpi=150)
plt.show()

## 10. Export Model for Production

In [ ]:
# Export as TorchScript for deployment
model.eval()

# Create sample input
sample_input = torch.randn(1, config.frames_per_video, 3, config.frame_size, config.frame_size).to(device)

# Script the model
scripted_model = torch.jit.trace(model, sample_input)

# Save
output_path = os.path.join(config.checkpoint_dir, 'video_deepfake_detector_scripted.pt')
scripted_model.save(output_path)
print(f"Saved TorchScript model to: {output_path}")

# Verify
loaded_model = torch.jit.load(output_path)
with torch.no_grad():
    test_output = loaded_model(sample_input)
    print(f"Test inference output shape: {test_output.shape}")
    print("✓ Model exported successfully!")

In [ ]:
# Download model files (for Colab)
from google.colab import files

# Download scripted model
files.download(output_path)

# Download training curves
files.download(os.path.join(config.checkpoint_dir, 'training_curves.png'))

## 11. Quick Inference Demo

In [ ]:
def predict_video(video_path, model, config):
    """
    Predict if a video is real or fake.
    """
    model.eval()
    
    # Create temporary dataset for single video
    temp_dataset = VideoDeepfakeDataset([video_path], [0], config)
    frames, _ = temp_dataset[0]
    frames = frames.unsqueeze(0).to(device)  # Add batch dimension
    
    with torch.no_grad():
        outputs = model(frames)
        probs = torch.softmax(outputs, dim=1)
        prediction = outputs.argmax(1).item()
        confidence = probs[0, prediction].item()
    
    label = "FAKE" if prediction == 1 else "REAL"
    
    print(f"Video: {os.path.basename(video_path)}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence*100:.2f}%")
    
    return label, confidence

# Test on a sample video
if len(test_paths) > 0:
    predict_video(test_paths[0], model, config)

---

## Next Steps

1. **Download the exported model** (`video_deepfake_detector_scripted.pt`)
2. **Copy to your project**: `ml/checkpoints/video_deepfake_detector_scripted.pt`
3. **Integrate with DeepGuard backend** for video detection endpoint

---

*Notebook created for DeepGuard Video Deepfake Detection*